# Optional B Test Helper - Create Sample New Incoming Developer Profile

Use this only when testing Pipeline B without a real new incoming raw-data batch.

It samples from `dev_profile_final_v4`, prefixes developer IDs, and writes `dev_profile_new_incoming_v1`.

This helper is **not** part of the default production Pipeline B shell script.


In [ ]:

import duckdb
from pathlib import Path

DB_PATH = "developer_project.duckdb"
SOURCE_PROFILE_TABLE = "dev_profile_final_v4"
NEW_PROFILE_TABLE = "dev_profile_new_incoming_v1"

# Number of developers to sample for testing.
# Increase this if you want a larger scoring test.
SAMPLE_N = 10_000

con = duckdb.connect(DB_PATH)
print("Connected to:", DB_PATH)


In [ ]:

def table_exists(con, table_name: str) -> bool:
    query = """
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = ?
    """
    return con.execute(query, [table_name]).fetchone()[0] > 0

missing = []
for table in [SOURCE_PROFILE_TABLE]:
    if not table_exists(con, table):
        missing.append(table)

if missing:
    raise ValueError(
        f"Missing required table(s): {missing}. "
        "Run 01_Creating_duckDB.ipynb, 02_Cleaning.ipynb, and 03_FeatureEngineering_v3.ipynb first."
    )

print("Source profile table found:", SOURCE_PROFILE_TABLE)



## Create sample new incoming profile table

This copies the same feature schema as `dev_profile_final_v4`, but samples only a subset of developers. The sampled developer IDs are prefixed with `NEW_SAMPLE_` so the rows behave like a separate incoming batch.


In [ ]:

source_cols = con.execute(f"DESCRIBE {SOURCE_PROFILE_TABLE}").df()["column_name"].astype(str).tolist()

if "developer_id" not in source_cols:
    raise ValueError(f"{SOURCE_PROFILE_TABLE} must contain developer_id")

select_exprs = []
for col in source_cols:
    safe_col = '"' + col.replace('"', '""') + '"'
    if col == "developer_id":
        select_exprs.append("'NEW_SAMPLE_' || CAST(developer_id AS VARCHAR) AS developer_id")
    else:
        select_exprs.append(safe_col)

select_sql = ",\n        ".join(select_exprs)

# DuckDB setseed expects a float in [-1, 1].
con.execute("SELECT setseed(0.42)")

create_sql = f"""
CREATE OR REPLACE TABLE {NEW_PROFILE_TABLE} AS
SELECT
        {select_sql}
FROM {SOURCE_PROFILE_TABLE}
USING SAMPLE {SAMPLE_N} ROWS
"""

try:
    con.execute(create_sql)
except Exception:
    con.execute(f"""
        CREATE OR REPLACE TABLE {NEW_PROFILE_TABLE} AS
        SELECT
                {select_sql}
        FROM {SOURCE_PROFILE_TABLE}
        ORDER BY random()
        LIMIT {SAMPLE_N}
    """)

print(f"Created or replaced table: {NEW_PROFILE_TABLE}")


In [ ]:

summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT developer_id) AS n_developers
FROM {NEW_PROFILE_TABLE}
""").df()

display(summary)

cols = con.execute(f"DESCRIBE {NEW_PROFILE_TABLE}").df()["column_name"].astype(str).tolist()
possible_stratum_cols = [c for c in ["stratum", "lifecycle_stratum", "lifecycle_status", "dormancy_status"] if c in cols]

for c in possible_stratum_cols:
    print(f"\nDistribution by {c}:")
    safe_c = '"' + c.replace('"', '""') + '"'
    display(con.execute(f"""
        SELECT {safe_c} AS {safe_c}, COUNT(*) AS n_developers
        FROM {NEW_PROFILE_TABLE}
        GROUP BY {safe_c}
        ORDER BY n_developers DESC
    """).df())



## Next step

Now run:

```bash
bash run_new_data_fixed_cluster_scoring_pipeline.sh
```

Or open and run:

```text
08_Score_New_Developers_LGBM_From_Fixed_HDBSCAN.ipynb
```

For real future data, do **not** use this sample notebook. Instead, create `dev_profile_new_incoming_v1` by running the same cleaning and feature engineering logic on the new raw data batch.


In [ ]:

con.close()
print("Done. Sample incoming profile table is ready for scoring.")
